In [ ]:
import lightning as L
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import wandb
import numpy as np
import random
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
train_data = pd.read_pickle("data/train.pkl")
test_data = pd.read_pickle("data/test_no_target.pkl")

In [ ]:
target_map = {0: "bach", 1: "beethoven", 2: "debussy", 3: "scarlatti", 4: "victoria"}

In [ ]:
train_data_df = pd.DataFrame(train_data, columns=["sequence", "target"])
train_data_df

In [ ]:
test_data_df = pd.Series(test_data, name="sequence").to_frame()
test_data_df

In [ ]:
target_counts = train_data_df["target"].value_counts()
target_counts.index = target_counts.index.map(target_map)
target_counts

In [ ]:
plt.figure(figsize=(10, 6))
target_counts.plot(kind="bar", color="skyblue")
plt.title("Distribution of Target Classes")
plt.xlabel("Target Class")
plt.ylabel("Count")
plt.grid(axis="y")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
sequence_lengths = train_data_df["sequence"].apply(lambda x: len(x))

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(sequence_lengths, bins=50, kde=True)
plt.title("Distribution of Sequence Lengths")
plt.xlabel("Sequence Length")
plt.ylabel("Frequency")
plt.axvline(
    sequence_lengths.mean(),
    color="r",
    linestyle="--",
    label=f"Mean: {sequence_lengths.mean():.2f}",
)
plt.axvline(
    sequence_lengths.median(),
    color="g",
    linestyle="-",
    label=f"Median: {sequence_lengths.median()}",
)
plt.legend()
plt.show()

In [ ]:
train_data_df

In [ ]:
def calculate_offset(series):
    min_val = int(series.apply(lambda x: min(x) if len(x) > 0 else 0).min())
    return 1 - min_val


def apply_offset_and_truncate(series, offset, max_len):
    return series.apply(lambda x: (x + offset)[:max_len])

In [ ]:
offset = calculate_offset(train_data_df["sequence"])
max_seq_len = int(train_data_df["sequence"].apply(len).quantile(0.95))

offset, max_seq_len

In [ ]:
train_data_df["processed_sequence"] = apply_offset_and_truncate(
    train_data_df["sequence"], offset, max_seq_len
)

In [ ]:
train_data_df

In [ ]:
def pad_collate(batch):
    xx = [torch.tensor(sample[0], dtype=torch.long) for sample in batch]
    yy = torch.tensor([sample[1] for sample in batch], dtype=torch.long)

    xx_pad = pad_sequence(xx, batch_first=True, padding_value=0)

    return xx_pad, yy

In [ ]:
class VariableLenDataset(Dataset):
    def __init__(self, in_data, target):
        self.data = [(x, y) for x, y in zip(in_data, target)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        in_data, target = self.data[idx]
        return in_data, target

In [ ]:
train_data_df

In [ ]:
unique_chords = set()
for sequence in train_data_df["processed_sequence"]:
    unique_chords.update(sequence.astype(int))

unique_chords = sorted(unique_chords)

In [ ]:
dataset = VariableLenDataset(
    in_data=train_data_df["processed_sequence"].tolist(),
    target=train_data_df["target"].tolist(),
)


In [ ]:
train_dataset, val_dataset = train_test_split(
    dataset, test_size=0.2, random_state=42, stratify=train_data_df["target"]
)

len(train_dataset), len(val_dataset)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
        num_classes: int,
        lstm_layers: int = 1,
        bidirectional: bool = True,
        dropout: float = 0.5,
        padding_idx: int = 0,
    ):
        super(LSTMClassifier, self).__init__()
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.lstm_layers = lstm_layers
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=padding_idx
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=lstm_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout,
        )

        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)

        if self.bidirectional:
            hidden_last_layer_forward = hidden[-2, :, :]
            hidden_last_layer_backward = hidden[-1, :, :]
            hidden_last_layer = torch.cat(
                (hidden_last_layer_forward, hidden_last_layer_backward), dim=1
            )
        else:
            hidden_last_layer = hidden[-1, :, :]

        out = self.fc(self.dropout(hidden_last_layer))

        return out

In [ ]:
from torchmetrics import Accuracy


class LSTMClassifierModule(L.LightningModule):
    def __init__(
        self,
        model_params: dict,
        optimizer_params: dict,
        class_weights=None,
    ):
        super(LSTMClassifierModule, self).__init__()

        self.save_hyperparameters()

        self.model_params = model_params
        self.optimizer_params = optimizer_params

        self.class_weights = class_weights

        self.model = LSTMClassifier(**model_params)

        self.criterion = nn.CrossEntropyLoss(weight=class_weights)

        self.train_acc = Accuracy(
            task="multiclass", num_classes=model_params["num_classes"]
        )
        self.val_acc = Accuracy(
            task="multiclass", num_classes=model_params["num_classes"]
        )

    def forward(self, x: torch.Tensor):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)

        self.train_acc(preds, y)
        self.log("train/loss", loss, prog_bar=True)
        self.log(
            "train/acc", self.train_acc, on_step=True, on_epoch=False, prog_bar=True
        )

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)

        self.val_acc(preds, y)
        self.log("val/loss", loss, prog_bar=True)
        self.log("val/acc", self.val_acc, on_step=False, on_epoch=True, prog_bar=True)

        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            params=self.model.parameters(), **self.optimizer_params
        )

        return {"optimizer": optimizer}

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=pad_collate,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=pad_collate,
)

In [ ]:
from lightning import Trainer
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping


checkpoint_callback = ModelCheckpoint(
    monitor="val/acc",
    mode="max",
    dirpath="checkpoints/",
    filename="best-model-{epoch:02d}-{val/acc:.2f}",
    save_top_k=1,
)
early_stopping_callback = EarlyStopping(
    monitor="val/acc",
    patience=10,
    mode="max",
)


wandb_logger = WandbLogger(project="chords-classification")

vocab_size = max(unique_chords) + 1

model_params = {
    "vocab_size": vocab_size,
    "embedding_dim": 64,
    "hidden_dim": 256,
    "num_classes": len(target_map),
    "lstm_layers": 2,
    "bidirectional": True,
    "dropout": 0.3,
}
optimizer_params = {
    "lr": 1e-3,
    "weight_decay": 1e-5,
}
model = LSTMClassifierModule(model_params, optimizer_params)

trainer = Trainer(
    max_epochs=50,
    accelerator="auto",
    logger=wandb_logger,
    callbacks=[checkpoint_callback, early_stopping_callback],
)

trainer.fit(model, train_loader, val_loader)

wandb.finish()

In [ ]:
model = LSTMClassifierModule.load_from_checkpoint(
    "checkpoints/light-elevator-27-epoch-26.ckpt"
)
model

In [ ]:
test_data_df["processed_sequence"] = apply_offset_and_truncate(
    pd.Series(test_data, name="sequence"), offset, max_seq_len
)
test_data_df

In [ ]:
test_dataset = VariableLenDataset(
    in_data=test_data_df["processed_sequence"].tolist(),
    target=[0] * len(test_data_df),  # Dummy targets for test set
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=pad_collate,
)

In [ ]:
model.eval()

predictions = []
with torch.inference_mode():
    for batch in test_loader:
        x, _ = batch
        x = x.to(model.device)
        logits = model(x)
        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy())


In [ ]:
assert len(predictions) == len(test_data_df), "Predictions length mismatch"

In [ ]:
predictions_df = pd.DataFrame(predictions, columns=["target"])
predictions_df

In [ ]:
predictions_df.to_csv("pred.csv", index=False, header=False)